In [0]:
%sql
CREATE CONNECTION IF NOT EXISTS earthquake_conn
type HTTP
OPTIONS (
  host = 'https://earthquake.usgs.gov',
  port = 443,
  base_path = '/earthquakes/feed/v1.0/',
  bearer_token = 'na'
)

In [0]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

conn = w.connections.get("earthquake_conn")
base_url = f"{conn.options['host']}{conn.options['base_path']}"

In [0]:
%sql
use catalog dev_cat;
use schema bronze;
create volume if not exists earthquake_data;


In [0]:
dbutils.widgets.text('catalog_name','dev_cat','dev_cat')
catalog_name=dbutils.widgets.get('catalog_name')
     

In [0]:

spark.sql(
    f"use catalog {catalog_name}"
)
spark.sql(
    "use schema bronze"
)
spark.sql(
    "create volume if not exists earthquake_data"
)

In [0]:
import requests
import json
import datetime

# Remove leading slash from the endpoint
url = f"{base_url}summary/all_day.geojson"  # Changed from /summary to summary
response = requests.get(url)
if response.status_code != 200:
    raise Exception(f"Error in getting data from {url}")
data = response.json()
current_date = datetime.datetime.now().strftime("%Y-%m-%d")
dbutils.fs.put(
    f"/Volumes/{catalog_name}/bronze/earthquake_data/earthquake_data_{current_date}.json",
    json.dumps(data),
    overwrite=True,
)

In [0]:
test_url = "https://earthquake.usgs.gov/earthquakes/feed/v1.0/summary/all_day.geojson"
response = requests.get(test_url)
print(f"Status: {response.status_code}")
print(f"Data count: {len(response.json()['features'])}")